# Final Clean Models

This notebook fits the final adopted Logistic Regression, Random Forest, and XGBoost configurations on the clean feature set.
It reuses the shared preprocessing pipeline, retrains on the combined train and validation split, evaluates on the held-out test split, and writes `results/main_clean.csv`.

In [1]:
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from xgboost import XGBClassifier

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline import build_pipeline, load_feature_frame, random_split

RANDOM_STATE = 42
data_path = project_root / 'data' / 'processed' / 'matches.parquet'
results_path = project_root / 'results' / 'main_clean.csv'

pd.set_option('display.max_columns', 50)

In [2]:
df_feat, y, _ = load_feature_frame(data_path, feature_set='clean')
splits = random_split(len(df_feat), y, test_size=0.20, val_size=0.125, seed=RANDOM_STATE)

trainval_idx = sorted(list(splits['train']) + list(splits['val']))
test_idx = splits['test']

X_trainval_raw = df_feat.iloc[trainval_idx].copy()
X_test_raw = df_feat.iloc[test_idx].copy()
y_trainval = y.iloc[trainval_idx].values
y_test = y.iloc[test_idx].values

pipeline = build_pipeline()
X_trainval = pipeline.fit_transform(X_trainval_raw)
X_test = pipeline.transform(X_test_raw)

pd.DataFrame(
    {
        'rows': [len(trainval_idx), len(test_idx)],
        'raw_cols': [df_feat.shape[1], df_feat.shape[1]],
        'matrix_cols': [X_trainval.shape[1], X_test.shape[1]],
    },
    index=['train_plus_val', 'test'],
)

,rows,raw_cols,matrix_cols
train_plus_val,80031,112,177
test,20008,112,177


In [3]:
final_model_params = {
    'LR': {'C': 0.1},
    'RF': {
        'n_estimators': 1100,
        'min_samples_leaf': 6,
        'max_features': 0.04,
        'max_depth': 28,
    },
    'XGB': {
        'subsample': 0.8,
        'reg_lambda': 8.0,
        'reg_alpha': 0.1,
        'n_estimators': 1400,
        'min_child_weight': 3,
        'max_depth': 3,
        'learning_rate': 0.06,
        'gamma': 0.3,
        'colsample_bytree': 0.75,
    },
}

final_model_meta = {
    'LR': {'adopted_round': 'phase2_round1', 'source_csv': 'best_params.csv'},
    'RF': {'adopted_round': 'phase2_round1', 'source_csv': 'best_params.csv'},
    'XGB': {'adopted_round': 'phase2_round4', 'source_csv': 'best_params_round4.csv'},
}

def evaluate_binary(model, X, y_true):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average='macro', zero_division=0
    )
    return {
        'test_roc_auc': roc_auc_score(y_true, proba),
        'test_accuracy': accuracy_score(y_true, pred),
        'test_macro_precision': precision,
        'test_macro_recall': recall,
        'test_macro_f1': f1,
    }

def make_models():
    return {
        'LR': LogisticRegression(
            solver='lbfgs',
            max_iter=1000,
            random_state=RANDOM_STATE,
            **final_model_params['LR'],
        ),
        'RF': RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **final_model_params['RF'],
        ),
        'XGB': XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            **final_model_params['XGB'],
        ),
    }

In [4]:
rows = []
fitted_models = {}

for model_name, model in make_models().items():
    start = perf_counter()
    model.fit(X_trainval, y_trainval)
    fit_seconds = perf_counter() - start
    fitted_models[model_name] = model

    rows.append(
        {
            'model': model_name,
            'feature_set': 'clean',
            'split': 'random',
            'fit_scope': 'train_plus_val',
            'seed': RANDOM_STATE,
            'raw_cols': df_feat.shape[1],
            'matrix_cols': X_trainval.shape[1],
            'trainval_rows': len(trainval_idx),
            'test_rows': len(test_idx),
            'adopted_round': final_model_meta[model_name]['adopted_round'],
            'source_csv': final_model_meta[model_name]['source_csv'],
            'best_params': str(final_model_params[model_name]),
            'fit_seconds': fit_seconds,
            **evaluate_binary(model, X_test, y_test),
        }
    )

main_clean = pd.DataFrame(rows).sort_values('model').reset_index(drop=True)
results_path.parent.mkdir(parents=True, exist_ok=True)
main_clean.to_csv(results_path, index=False)

print(f'Wrote {results_path}')
main_clean

Wrote /mnt/ncsudrive/x/xzhou38/CSC522-S26-proj-deadlock-match-forecast/results/main_clean.csv


,model,feature_set,split,fit_scope,seed,raw_cols,matrix_cols,trainval_rows,test_rows,adopted_round,source_csv,best_params,fit_seconds,test_roc_auc,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1
0,LR,clean,random,train_plus_val,42,112,177,80031,20008,phase2_round1,best_params.csv,{'C': 0.1},3.677251,0.743801,0.680428,0.680439,0.680287,0.680292
1,RF,clean,random,train_plus_val,42,112,177,80031,20008,phase2_round1,best_params.csv,"{'n_estimators': 1100, 'min_samples_leaf': 6, ...",25.569713,0.727773,0.666883,0.667157,0.666595,0.666483
2,XGB,clean,random,train_plus_val,42,112,177,80031,20008,phase2_round4,best_params_round4.csv,"{'subsample': 0.8, 'reg_lambda': 8.0, 'reg_alp...",5.087570,0.748676,0.683627,0.683637,0.683489,0.683496
